# Horizon-Money Track 4: `macroTomicro` Google Colab A100 GPU Offload & AFML Auditor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecopeko/horizon-money-studio/blob/main/colab/track4_m2m_gpu_offload_sweep.ipynb)

This notebook runs heavy walk-forward PyTorch neural vs `DayBlockBaggingRegressor` sweeps and AFML Ch.4/6/8 orthogonality audits on **Colab A100 / L4** using your monthly **2,000 Colab CCUs**, keeping the live Windows host (`RTX 4060 Ti 8GB`, `8 GB` free RAM) completely unblocked.

**Aligned Capital & Audit Rules (`macroTomicro` AGENTS.md & DAILY_RESEARCH_EVALUATION.md):**
- **Japan Equities**: `800,000 JPY` base capital, `2,400,000 JPY` fixed margin cap, `100`-share lot truncation, daily independent non-compounding reset (`LONG 5bp`, `SHORT 35bp`).
- **US Equities**: `$228.20 USD` Webull cash (`$221.55 USD` 103% market-order cap), `1`-share integer truncation, `top10_ranked_affordable_long` selection, non-compounding (`5bp`).
- **Zero Fake Fallbacks & Uniform-Shrinkage Controls**: Never `.fillna(0)` on missing returns/prices; every filtering/veto policy is benchmarked against a uniform-shrinkage control (`対一律縮小対照`).
- **AFML Ch.4/6/8 Audits**: `DayBlockBaggingRegressor`, In-Sample MDI/MDA vs OOS MDA permutation drop, and Kaiser PCA Top-K vs Bottom-K orthogonality verification.

In [ ]:
#@title 1. 1-Click Setup: Clone `horizon-money-studio`, Optional Drive Mount & Load AFML Auditor
!git clone -q https://github.com/pecopeko/horizon-money-studio.git /content/horizon-money-studio || git -C /content/horizon-money-studio pull -q

import sys
from pathlib import Path
import torch

REPO_DIR = Path('/content/horizon-money-studio')
for p in (str(REPO_DIR), str(REPO_DIR / 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

MOUNT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
PANEL_DIR = Path('/content/horizon_m2m_panels')
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_panel_dir = Path('/content/drive/MyDrive/horizon_money/m2m_panels')
        drive_panel_dir.mkdir(parents=True, exist_ok=True)
        PANEL_DIR = drive_panel_dir
    except Exception as exc:
        print(f'Drive mount skipped/unavailable ({exc}); using local workspace {PANEL_DIR}')

PANEL_DIR.mkdir(parents=True, exist_ok=True)

from tracks.track4_m2m_colab.afml_colab_auditor import (
    DayBlockBaggingRegressor,
    evaluate_jp_daily_fixed_capital,
    evaluate_us_daily_fixed_capital,
    run_afml_orthogonality_audit,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU fallback'
print(f'afml_colab_auditor loaded | Device: {DEVICE} ({gpu_name}) | PANEL_DIR: {PANEL_DIR}')

In [ ]:
#@title 2. Walk-Forward Sweep: GPU PyTorch TabularResMLP vs DayBlockBaggingRegressor vs Ridge
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

PANEL_PATH = str(PANEL_DIR / 'panel_stationary.parquet')  #@param {type:'string'}
DATE_COL = 'signal_date'  #@param {type:'string'}
TARGET_COL = 'target_bp'  #@param {type:'string'}
REFIT_STEP_DAYS = 20  #@param {type:'integer'}
MLP_EPOCHS = 40  #@param {type:'integer'}
MLP_SEEDS = 3  #@param {type:'integer'}

def _build_demo_panel(n_days: int = 180, rows_per_day: int = 8) -> tuple[pd.DataFrame, list[str]]:
    rng = np.random.default_rng(42)
    dates = pd.bdate_range('2025-04-01', periods=n_days).strftime('%Y-%m-%d')
    rows = []
    for idx_d, d in enumerate(dates):
        macro = rng.normal(0.0, 1.0)
        for s in range(rows_per_day):
            f_alpha = rng.normal(0.0, 1.0)
            f_collinear = f_alpha + rng.normal(0.0, 0.15)
            f_macro_proxy = macro + rng.normal(0.0, 0.25)
            f_regime_vol = abs(macro) + rng.normal(0.0, 0.2)
            f_noise_a = rng.normal(0.0, 1.0)
            # Non-linear interaction + linear drift + realistic overnight bp scale
            target_bp = 45.0 * f_alpha + 25.0 * f_macro_proxy + 18.0 * np.tanh(f_alpha * f_macro_proxy) + rng.normal(0.0, 85.0)
            # Train-only spurious leakage only in first 60% of sample to verify AFML MDA detection
            f_spurious = (0.008 * target_bp + rng.normal(0.0, 0.4)) if idx_d < int(n_days * 0.6) else rng.normal(0.0, 1.0)
            px = round(1500.0 + 400.0 * rng.random(), 1)
            rows.append({
                DATE_COL: d,
                'code': f'DEMO_{s:02d}',
                'entry_price': px,
                'exit_price': round(px * (1.0 + target_bp / 10_000.0), 2),
                TARGET_COL: target_bp,
                'f_alpha': f_alpha,
                'f_collinear': f_collinear,
                'f_macro_proxy': f_macro_proxy,
                'f_regime_vol': f_regime_vol,
                'f_noise_a': f_noise_a,
                'f_spurious_train_only': f_spurious,
            })
    frame = pd.DataFrame(rows)
    feats = [c for c in frame.columns if c.startswith('f_')]
    return frame, feats

if Path(PANEL_PATH).exists():
    df = pd.read_parquet(PANEL_PATH)
    feature_cols = [c for c in df.columns if c.startswith('f_')]
    df = df.dropna(subset=[DATE_COL, TARGET_COL] + feature_cols).copy()
    df[DATE_COL] = df[DATE_COL].astype(str).str.slice(0, 10)
    df = df.sort_values(DATE_COL).reset_index(drop=True)
    data_source = f'Loaded real Drive/local panel: {PANEL_PATH}'
else:
    df, feature_cols = _build_demo_panel()
    data_source = 'No custom panel_stationary.parquet found; running realistic synthetic multi-day AFML + GPU benchmark.'

print(data_source)
print(f'Rows={len(df)} | Unique dates={df[DATE_COL].nunique()} | Features={len(feature_cols)}')

class _ResMLPNet(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 64) -> None:
        super().__init__()
        self.in_proj = nn.Linear(in_dim, hidden)
        self.block = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Dropout(0.15),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
        )
        self.head = nn.Linear(hidden, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.in_proj(x)
        h = h + self.block(h)
        return self.head(h).squeeze(-1)

class TorchResMLPEnsemble(BaseEstimator, RegressorMixin):
    """Multi-seed GPU PyTorch Residual MLP compatible with run_afml_orthogonality_audit."""

    def __init__(self, hidden: int = 64, epochs: int = 40, lr: float = 2e-3, n_seeds: int = 3, base_seed: int = 42) -> None:
        self.hidden = hidden
        self.epochs = epochs
        self.lr = lr
        self.n_seeds = n_seeds
        self.base_seed = base_seed

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'TorchResMLPEnsemble':
        X_arr = np.asarray(X, dtype=np.float32)
        y_arr = np.asarray(y, dtype=np.float32)
        self.y_scale_ = max(float(np.std(y_arr)), 1e-6)
        self.y_mean_ = float(np.mean(y_arr))
        y_norm = (y_arr - self.y_mean_) / self.y_scale_

        xt = torch.tensor(X_arr, device=DEVICE)
        yt = torch.tensor(y_norm, device=DEVICE)
        self.nets_ = []

        for s in range(self.n_seeds):
            torch.manual_seed(self.base_seed + s)
            net = _ResMLPNet(X_arr.shape[1], self.hidden).to(DEVICE)
            opt = torch.optim.AdamW(net.parameters(), lr=self.lr, weight_decay=1e-2)
            loss_fn = nn.HuberLoss(delta=1.5)
            net.train()
            for _ in range(self.epochs):
                opt.zero_grad()
                loss = loss_fn(net(xt), yt)
                loss.backward()
                opt.step()
            net.eval()
            self.nets_.append(net)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        X_arr = np.asarray(X, dtype=np.float32)
        xt = torch.tensor(X_arr, device=DEVICE)
        preds = []
        with torch.no_grad():
            for net in self.nets_:
                preds.append(net(xt).cpu().numpy() * self.y_scale_ + self.y_mean_)
        return np.mean(preds, axis=0)

unique_dates = sorted(df[DATE_COL].unique())
split_idx = int(len(unique_dates) * 0.60)
oos_date_chunks = [unique_dates[i:i + REFIT_STEP_DAYS] for i in range(split_idx, len(unique_dates), REFIT_STEP_DAYS)]

oos_records = []
for chunk in oos_date_chunks:
    first_d = chunk[0]
    tr_df = df[df[DATE_COL] < first_d]
    te_df = df[df[DATE_COL].isin(chunk)].copy()
    scaler = StandardScaler().fit(tr_df[feature_cols].to_numpy())
    X_tr = scaler.transform(tr_df[feature_cols].to_numpy())
    y_tr = tr_df[TARGET_COL].to_numpy(dtype=float)
    d_tr = tr_df[DATE_COL].to_numpy()
    X_te = scaler.transform(te_df[feature_cols].to_numpy())

    ridge = Ridge(alpha=50.0).fit(X_tr, y_tr)
    bagger = DayBlockBaggingRegressor(n_estimators=25, alpha=50.0, random_state=42).fit(X_tr, y_tr, dates=d_tr)
    mlp = TorchResMLPEnsemble(epochs=MLP_EPOCHS, n_seeds=MLP_SEEDS).fit(X_tr, y_tr)

    te_df['pred_ridge'] = ridge.predict(X_te)
    te_df['pred_bag'] = bagger.predict(X_te)
    te_df['pred_mlp'] = mlp.predict(X_te)
    te_df['pred_blend'] = 0.5 * te_df['pred_bag'] + 0.5 * te_df['pred_mlp']
    oos_records.append(te_df)

oos_df = pd.concat(oos_records, ignore_index=True)
for col in ['pred_ridge', 'pred_bag', 'pred_mlp', 'pred_blend']:
    ic = float(np.corrcoef(oos_df[TARGET_COL], oos_df[col])[0, 1])
    sp = float(spearmanr(oos_df[TARGET_COL], oos_df[col]).statistic)
    print(f'WF OOS {col:12s} | Pearson IC: {ic:+.4f} | Spearman IC: {sp:+.4f} | n={len(oos_df)}')

In [ ]:
#@title 3. AFML Ch.8 Orthogonality Audit (MDI/Train-MDA vs OOS MDA + Kaiser PCA)
split_date = unique_dates[split_idx]
tr_static = df[df[DATE_COL] < split_date]
te_static = df[df[DATE_COL] >= split_date]

static_scaler = StandardScaler().fit(tr_static[feature_cols].to_numpy())
X_tr_s = static_scaler.transform(tr_static[feature_cols].to_numpy())
X_te_s = static_scaler.transform(te_static[feature_cols].to_numpy())
y_tr_s = tr_static[TARGET_COL].to_numpy(dtype=float)
y_te_s = te_static[TARGET_COL].to_numpy(dtype=float)
d_tr_s = tr_static[DATE_COL].to_numpy()

bag_audit = run_afml_orthogonality_audit(
    DayBlockBaggingRegressor(n_estimators=25, alpha=50.0, random_state=42),
    X_tr_s, y_tr_s, d_tr_s, X_te_s, y_te_s, feature_cols, random_state=42, n_repeats=15,
)
mlp_audit = run_afml_orthogonality_audit(
    TorchResMLPEnsemble(epochs=MLP_EPOCHS, n_seeds=MLP_SEEDS),
    X_tr_s, y_tr_s, None, X_te_s, y_te_s, feature_cols, random_state=42, n_repeats=15,
)

for name, aud in [('DayBlockBaggingRegressor', bag_audit), ('TorchResMLPEnsemble', mlp_audit)]:
    print(f'=== {name} ===')
    print(f'Train IC={aud.train_pearson_ic:+.4f} | OOS Pearson={aud.oos_pearson_ic:+.4f} | OOS Spearman={aud.oos_spearman_ic:+.4f}')
    print(f'Kaiser dims={aud.kaiser_effective_dims}/{aud.total_features} | PCA Top-K={aud.pca_top_k_oos_ic:+.4f} vs Bot-K={aud.pca_bottom_k_oos_ic:+.4f} | edge_on_top_pcs={aud.edge_on_top_pcs}')
    print('Flagged Overfit Features:', aud.flagged_overfit_features)
    display_df = pd.DataFrame([r.model_dump() for r in aud.feature_audit]).sort_values('oos_mda_ic', ascending=False)
    print(display_df.to_string(index=False))

In [ ]:
#@title 4. Non-Compounding Fixed-Capital Backtest + Uniform-Shrinkage Control & OOS Parquet Export
# Pick top-1 ranked candidate per day by pred_blend and compare AlwaysTrade vs pred_blend > 0 Cash Veto
oos_ranked = oos_df.sort_values([DATE_COL, 'pred_blend'], ascending=[True, False]).groupby(DATE_COL).first().reset_index()
n_oos_days = len(oos_ranked)

if 'entry_price' in oos_ranked.columns and 'exit_price' in oos_ranked.columns:
    all_jp_trades = [
        {'date': r[DATE_COL], 'side': 'LONG', 'entry_price': float(r['entry_price']), 'exit_price': float(r['exit_price'])}
        for _, r in oos_ranked.iterrows()
    ]
    gated_jp_trades = [
        {'date': r[DATE_COL], 'side': 'LONG', 'entry_price': float(r['entry_price']), 'exit_price': float(r['exit_price'])}
        for _, r in oos_ranked.iterrows() if float(r['pred_blend']) > 0.0
    ]
    rep_all = evaluate_jp_daily_fixed_capital(all_jp_trades, total_business_days=n_oos_days)
    rep_gate = evaluate_jp_daily_fixed_capital(gated_jp_trades, total_business_days=n_oos_days)
    active_rate = rep_gate.trade_count / max(rep_all.trade_count, 1)
    rep_unif = evaluate_jp_daily_fixed_capital(
        all_jp_trades, margin_cap_jpy=2_400_000.0 * active_rate, total_business_days=n_oos_days
    )
    print('JP Fixed-Cap AlwaysTrade (800k/2.4M):', rep_all.total_net_pnl, 'JPY | Sharpe:', rep_all.annualized_sharpe)
    print(f'JP Fixed-Cap Blend>0 Veto ({rep_gate.trade_count}/{n_oos_days}):', rep_gate.total_net_pnl, 'JPY | Sharpe:', rep_gate.annualized_sharpe)
    print(f'JP Uniform-Shrinkage Control ({active_rate:.1%} cap):', rep_unif.total_net_pnl, 'JPY | Sharpe:', rep_unif.annualized_sharpe)

# US Webull $228.20 / $221.55 cap example (top10_ranked_affordable_long)
sample_us_days = {
    '2026-11-16': [
        {'symbol': 'EXPENSIVE', 'entry_price': 450.0, 'exit_price': 460.0},
        {'symbol': 'AFFORDABLE', 'entry_price': 35.0, 'exit_price': 36.4},
    ],
    '2026-11-17': [],
}
us_report = evaluate_us_daily_fixed_capital(sample_us_days)
print('US Webull Sanity Check ($228.20 / $221.55 cap):', us_report.total_net_pnl, 'USD | JPY:', us_report.total_net_pnl_jpy)

oos_out_path = PANEL_DIR / 'colab_wf_oos_predictions.parquet'
oos_df[[DATE_COL, TARGET_COL, 'pred_ridge', 'pred_bag', 'pred_mlp', 'pred_blend']].to_parquet(oos_out_path, index=False)
print('Exported compact OOS prediction cache ->', oos_out_path)